<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/sinhala_textbook_extraction/textbook_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setup

*   Installs pymupdf (to read the PDF’s text) and pandukabhaya (to convert FM-Abhaya to Unicode), then connects Drive.

In [ ]:
!pip install -q pymupdf pandukabhaya

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 41.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, random, subprocess, unicodedata, datetime
from collections import Counter
from pandukabhaya import Converter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Load the book

In [ ]:
BASE    = "/content/drive/MyDrive/Research-WSD-New"
PDF_DIR = f"{BASE}/textbooks"
OUT_DIR = f"{BASE}/textbooks/Ext_Final"

BOOKS = [
    {
        "book_id":     "sinhala_lit_g10_11",
        "pdf":         "sinhala lit g-10-11.pdf",
        "title":       "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
        "publisher":   "Educational Publications Department",
        "grade":       "10-11",
        "year":        "2020",
        "edition":     "7th printing (first printed 2014)",
        "isbn":        "978-955-25-0381-8",
        "url":         "http://edupub.gov.lk/Administrator/Sinhala/10/sin%20lit%20g-10-11/sinhala%20lit%20g-10-11.pdf",
        "page_offset": 10,
    },
]
for b in BOOKS:
    b["pdf_path"] = os.path.join(PDF_DIR, b["pdf"])
    b["out"] = os.path.join(OUT_DIR, b["book_id"]); os.makedirs(b["out"], exist_ok=True)
    print(("OK      " if os.path.exists(b["pdf_path"]) else "MISSING ") + b["pdf_path"])

OK      /content/drive/MyDrive/Research-WSD-New/textbooks/sinhala lit g-10-11.pdf


3. Read the text and check its encoding

In [ ]:
# Why not pdftotext: (1) it reads two-column poems column by column, so a verse line loses its
# rhyme word; (2) it can't see fonts, so English words (Times font) were "converted" into fake Sinhala.
# PyMuPDF gives each text span with its font and x/y position, so we can fix both.
import pymupdf
COL_GAP, WORD_GAP, LINE_TOL = 12.0, 1.5, 3.0

def read_spans(path):
    """List of pages; each page = list of spans {text, font, bbox}."""
    with pymupdf.open(path) as doc:
        return [[s for b in p.get_text("dict")["blocks"] for l in b.get("lines", []) for s in l["spans"]
                 if s["text"].strip()] for p in doc]

def detect_encoding(pages):
    """'fm' if most text is in FM (legacy) fonts, 'unicode' if it is already Sinhala Unicode,
    'none' if there is (almost) no text layer -> scanned -> needs Document AI OCR."""
    chars = Counter()
    for p in pages:
        for s in p:
            chars["fm" if s["font"].startswith("FM") else
                  "unicode" if SINHALA.search(s["text"]) else "other"] += len(s["text"])
    total = sum(chars.values())
    if total < 500:
        return "none", chars
    return ("fm" if chars["fm"] > chars["unicode"] else "unicode"), chars

SINHALA = re.compile(r"[\u0D80-\u0DFF]")
LATIN   = re.compile(r"[A-Za-z]")
for b in BOOKS:
    b["spans"] = read_spans(b["pdf_path"])
    b["encoding"], chars = detect_encoding(b["spans"])
    fonts = Counter(s["font"] for p in b["spans"] for s in p)
    print(f'{b["book_id"]}: {len(b["spans"])} pages -> {b["encoding"]} | chars by type {dict(chars)}')
    print("   fonts:", fonts.most_common(8))

sinhala_lit_g10_11: 140 pages -> fm | chars by type {'fm': 224027, 'other': 1912}
   fonts: [('FMAbhayax', 6420), ('TimesNewRomanPSMT', 1259), ('FMAbabldBold', 446), ('MinionPro-Regular', 144), ('FMEmaneex', 70), ('FMSamanthaBoldx', 16), ('TimesNewRomB', 4), ('TimesNewRomanPS-BoldMT', 3)]


4. Convert FM-Abhaya to Unicode

In [ ]:
conv = Converter("fm_abhaya")
NOISE = re.compile(r"[\u00AD\u200B\uFEFF]")

# Glyphs the fm_abhaya table leaves unconverted in this book (checked on real words):
# `ඵ -> ළු (නෙළුම්, පුළුවන්), `ද -> ඳ (අඳින්නේ), `ඩ -> ඬ (පඬිතුමා), ¥ -> දූ (දූතයා, දූවිලි),
# i after a Sinhala letter -> ද්‍ය (විද්‍යාලය, පද්‍ය).
POST_FIXES = [(re.compile(r"`ඵ"), "ළු"), (re.compile(r"`ද"), "ඳ"), (re.compile(r"`ඩ"), "ඬ"),
              (re.compile(r"¥"), "දූ"), (re.compile(r"(?<=[\u0D80-\u0DFF])i"), "ද්\u200Dය")]

def fm_to_unicode(t):
    # “ and ‘ are used as quote marks in this book, but the table maps “ -> ර්‍ණ and ' ‘' -> ි.
    # Protect them BEFORE converting so real letters are untouched.
    t = t.replace("“", "\u0001").replace("‘", "\u0002")
    t = conv.convert(t).replace("\u0001", "“").replace("\u0002", "‘")
    for pat, rep in POST_FIXES:
        t = pat.sub(rep, t)
    return t

def segment_text(spans, encoding):
    """Text of spans that sit close together on one line. FM-font runs are converted together;
    other fonts (Times, Minion = English, ISBN) are kept exactly as they are."""
    out, run, run_fm, prev_x1 = [], "", None, None
    for s in spans:
        fm = encoding == "fm" and s["font"].startswith("FM")
        txt = s["text"]
        if prev_x1 is not None and s["bbox"][0] - prev_x1 > WORD_GAP and not txt.startswith(" "):
            txt = " " + txt
        if run_fm is None or fm == run_fm:
            run += txt; run_fm = fm
        else:
            out.append(fm_to_unicode(run) if run_fm else run); run, run_fm = txt, fm
        prev_x1 = s["bbox"][2]
    if run:
        out.append(fm_to_unicode(run) if run_fm else run)
    return re.sub(r"\s+", " ", "".join(out)).strip()

DEP_SIGN = re.compile(r"^[\u0DCA-\u0DDF\u0D82\u0D83]")
NUM_ONLY = re.compile(r"^\(?\d{1,3}[.)]?$")

GLOSS_ROW = re.compile(r"^\S[^.?!]{0,60}\s-\s")
is_short = lambda t: all(len(w) <= 2 for w in t.split())

def merge_segments(segs):
    """Pieces of one printed line (left to right, already without a second column) -> (lines, new_unit).
    - rhyme word / syllables printed apart on the right of a verse line -> joined to the line
      (ගොදු + රු = ගොදුරු, පඳුරු + ත ළා = පඳුරු තළා)
    - glossary 'word   - meaning' -> one row; two glossary entries on one printed line -> two rows
    - label 'අන්වය :   text' -> one line
    A stanza / question number at the start ("28.") or a glossary row starts a new unit."""
    new_unit = False
    if len(segs) > 1 and NUM_ONLY.match(segs[0]):
        segs, new_unit = segs[1:], True
    out, cur = [], segs[0]
    for i, nxt in enumerate(segs[1:], start=1):
        next_is_meaning = i + 1 < len(segs) and segs[i + 1].startswith("-")
        if " - " in cur and next_is_meaning:
            out.append(cur); cur = nxt
        elif DEP_SIGN.match(nxt) or (is_short(nxt) and len(cur.split()[-1]) <= 1):
            cur += nxt.replace(" ", "")
        elif is_short(nxt):
            cur += " " + nxt.replace(" ", "")
        else:
            cur += " " + nxt
    out.append(cur)
    return out, new_unit or bool(GLOSS_ROW.match(out[0]))

def rebuild_page(spans, encoding):
    """Group spans into printed lines (same baseline), split lines at big gaps, merge, and return the
    page text: left column first, then the right column (only where a page really has two)."""
    spans = sorted(spans, key=lambda s: (round(s["bbox"][3]), s["bbox"][0]))
    kept = []                                    # drop "fake bold" duplicates: same text printed twice
    for s in spans:                              # a few points apart (shadow effect)
        if not any(k["text"].replace(" ", "") == s["text"].replace(" ", "") and abs(k["bbox"][0] - s["bbox"][0]) < 5
                   and abs(k["bbox"][3] - s["bbox"][3]) < 4 for k in kept[-6:]):
            kept.append(s)
    spans, rows = kept, []
    for s in spans:
        if rows and abs(rows[-1][0] - s["bbox"][3]) <= LINE_TOL:
            rows[-1][1].append(s)
        else:
            rows.append([s["bbox"][3], [s]])
    # pass 1: cut every printed line into pieces at big gaps, remembering where each piece starts (x)
    lines = []
    for _, ss in rows:
        ss.sort(key=lambda s: s["bbox"][0])
        groups, cur = [], [ss[0]]
        for s in ss[1:]:
            if s["bbox"][0] - cur[-1]["bbox"][2] > COL_GAP:
                groups.append(cur); cur = [s]
            else:
                cur.append(s)
        groups.append(cur)
        segs = [(segment_text(g, encoding), g[0]["bbox"][0]) for g in groups]
        lines.append([(t, x) for t, x in segs if t])
    # pass 2: find a real two-column block (e.g. a two-column song). A "right-column start" is a piece
    # that begins right of 45% of the page width and is several real words — not a rhyme word, not
    # spaced rhyme syllables, not a glossary meaning. If a page has 4+ such lines, every piece that
    # starts at or right of that column, between the first and last such line, goes to column 2.
    width = max((s["bbox"][2] for s in spans), default=0)
    def col2_start(t, x):
        return x > 0.45 * width and " " in t and not is_short(t) and not t.startswith("-")
    hits = [i for i, segs in enumerate(lines)
            if any(col2_start(t, x) for t, x in segs) and not any(t.startswith("-") for t, _ in segs)]
    if len(hits) >= 4:
        boundary = min(x for i in hits for t, x in lines[i] if col2_start(t, x)) - 10
        lo, hi = hits[0], hits[-1]
    else:
        boundary, lo, hi = width + 1, -1, -1
    streams = {}
    for i, segs in enumerate(lines):
        if not segs:
            continue
        in_block = lo <= i <= hi
        to_col2 = lambda t, x: in_block and x >= boundary and not is_short(t)   # rhyme syllables stay
        left = [t for t, x in segs if not to_col2(t, x)]
        rcol = " ".join(t for t, x in segs if to_col2(t, x))
        rows, new_unit = merge_segments(left) if left else ([], False)
        targets = [(0, t) for t in rows] + ([(1, rcol)] if rcol else [])
        for col, text in targets:
            st = streams.setdefault(col, [])
            if new_unit or (col == 0 and GLOSS_ROW.match(text)):
                st.append("")                      # blank line = hard boundary before this line
            st.append(text)
            if col == 0 and GLOSS_ROW.match(text):
                st.append("")
    page = "\n\n".join("\n".join(streams[k]) for k in sorted(streams))
    return NOISE.sub("", unicodedata.normalize("NFC", page))

for b in BOOKS:
    if b["encoding"] == "none":
        print(f'{b["book_id"]}: no text layer — skip here, use Document AI OCR'); continue
    b["pages"] = [rebuild_page(p, b["encoding"]) for p in b["spans"]]
    json.dump(b["pages"], open(f'{b["out"]}/{b["book_id"]}_pages_unicode.json', "w", encoding="utf-8"), ensure_ascii=False)
    print(f'{b["book_id"]}: {len(b["pages"])} pages converted')
    print("   sample:", re.sub(r"\s+", " ", b["pages"][min(40, len(b["pages"])-1)])[:200])

sinhala_lit_g10_11: 140 pages converted
   sample: (2) වනපෙතිනැවිදින් : වනපෙතින් + ඇවිදින් = වනපෙතිනැවිදින් (3) සුමුවෝ : සමූහ ව (4) නදළ මුවෝ : නද + අළ මුවෝ = නදළ මුවෝ අළ යනු අර (ඇරීමේ) ධාතුවේ අතීත කෘදන්තයි. නාදයට විස්මිත ව කට ඇර ගෙන සිටියා වූ අළ මුව ය


5. Check the conversion page by page

In [ ]:
# After a good conversion almost no Latin letters remain. Leftover Latin = glyphs the converter
# did not map (or real English text on that page) have to look at those pages before using them.
LATIN_LEFT_MAX = 0.02          # flag a page if > 2% of its letters are still Latin

rows = []
for b in BOOKS:
    if "pages" not in b: continue
    leftovers = Counter()
    for i, p in enumerate(b["pages"]):
        s, l = len(SINHALA.findall(p)), len(LATIN.findall(p))
        leftovers.update(ch for ch in p if not (SINHALA.match(ch) or ch.isspace() or ch.isdigit()
                                               or ch in ".,;:!?()[]-–—'\"“”‘’/\u200C\u200D"))
        rows.append({"book": b["book_id"], "pdf_page": i + 1, "sinhala": s, "latin": l,
                     "flag": (s + l) > 0 and l / (s + l) > LATIN_LEFT_MAX})
    flagged = [r["pdf_page"] for r in rows if r["book"] == b["book_id"] and r["flag"]]
    print(f'{b["book_id"]}: {len(flagged)} flagged pages {flagged[:20]}')
    print("   most common leftover characters:", leftovers.most_common(15))

import pandas as pd
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/conversion_page_check.csv", index=False, encoding="utf-8")

sinhala_lit_g10_11: 6 flagged pages [2, 4, 5, 8, 53, 54]
   most common leftover characters: [('i', 35), ('+', 30), ('e', 28), ('n', 24), ('a', 23), ('l', 17), ('t', 17), ('h', 12), ('=', 12), ('u', 11), ('o', 11), ('r', 11), ('g', 10), ('s', 8), ('d', 7)]


6. page exclusions

In [ ]:
# Pages to leave out of the dataset:
#  - front matter: every PDF page before printed page 1 (covers, preface, contents)
#  - pages you checked by eye and found badly converted (set after looking at flagged pages)
MANUAL_EXCLUDE = {53, 54}       # <- change after you check pages 53 and 54; use set() if they are fine
for b in BOOKS:
for b in BOOKS:
    b["exclude_pages"] = set(range(1, b["page_offset"] + 1)) | MANUAL_EXCLUDE
    print(b["book_id"], "excluded PDF pages:", sorted(b["exclude_pages"]))

7. Split into sentences


*   Page-number lines are dropped.
*   Prose is split at . ? !.
*   Long poem chunks are cut into 2-line units, because poems have no full stops.

In [ ]:
#  - lines that are only page numbers (digits / roman numerals) are dropped
#  - the book's study notes start with labels අන්වය : / අර්ථය : / විස්තර : -> each label starts a new unit,
#    so a verse stanza is never glued to its explanation
#  - prose is split after . ? !
#  - verse has no full stops: a chunk > 280 chars with >= 3 lines is cut into units of
#    VERSE_LINES lines
#  - units with fewer than 4 Sinhala letters are dropped (numbering "03.", initials "එම්.")
VERSE_LINES = 2
PAGE_NUM = re.compile(r"[\divxlcIVXLC\s]+")
SPLIT    = re.compile(r"(?<=[.?!])\s+")
LABEL    = re.compile(r"\s*(?=(?:අන්වය|අර්ථය|විස්තර)\s*\*?\s*:)")

def split_units(pages, offset):
    units = []
    for pi, p in enumerate(pages):
        lines = [re.sub(r"[ \t]+", " ", l).strip() for l in p.split("\n")]
        lines = [l for l in lines if not PAGE_NUM.fullmatch(l)]
        text = LABEL.sub("\n\n", "\n".join(lines))
        text = re.sub(r"\n{3,}", "\n\n", text)
        for block in text.split("\n\n"):
            for s in SPLIT.split(block):
                s = s.strip()
                if not s: continue
                if len(s) > 280 and s.count("\n") >= 2:
                    ls = s.split("\n")
                    chunks = [" ".join(ls[i:i + VERSE_LINES]) for i in range(0, len(ls), VERSE_LINES)]
                else:
                    chunks = [s.replace("\n", " ")]
                for c in chunks:
                    c = re.sub(r"\s+", " ", c).strip()
                    if len(SINHALA.findall(c)) >= 4:
                        bp = pi + 1 - offset
                        units.append({"sid": len(units), "pdf_page": pi + 1,
                                      "book_page": bp if bp >= 1 else "", "text": c})
    return units

for b in BOOKS:
    if "pages" not in b: continue
    b["units"] = split_units(b["pages"], b["page_offset"])
    with open(f'{b["out"]}/{b["book_id"]}_sentences.jsonl', "w", encoding="utf-8") as f:
        for u in b["units"]:
            f.write(json.dumps({"book_id": b["book_id"], **u}, ensure_ascii=False) + "\n")
    with open(f'{b["out"]}/{b["book_id"]}_corpus.txt', "w", encoding="utf-8") as f:   # one unit per line
        f.write("\n".join(u["text"] for u in b["units"]))
    print(f'{b["book_id"]}: {len(b["units"]):,} sentence units -> {b["book_id"]}_corpus.txt')

sinhala_lit_g10_11: 3,190 sentence units -> sinhala_lit_g10_11_corpus.txt


7. Spot-check sheet

In [ ]:
SPOT_N, SEED = 100, 42
for b in BOOKS:
    if "units" not in b: continue
    sample = random.Random(SEED).sample(b["units"], min(SPOT_N, len(b["units"])))
    pd.DataFrame([{"sid": u["sid"], "pdf_page": u["pdf_page"], "book_page": u["book_page"],
                   "text": u["text"], "ok (y/n)": "", "error type": "", "note": ""}
                  for u in sorted(sample, key=lambda u: u["sid"])]
                 ).to_excel(f'{b["out"]}/{b["book_id"]}_spot_check.xlsx', index=False)
    print(f'{b["book_id"]}: {len(sample)} units -> {b["book_id"]}_spot_check.xlsx')

sinhala_lit_g10_11: 100 units -> sinhala_lit_g10_11_spot_check.xlsx


In [ ]:
import importlib.metadata as md
for b in BOOKS:
    if "units" not in b: continue
    meta = {k: b[k] for k in ["book_id", "pdf", "title", "publisher", "grade", "year", "url", "page_offset", "encoding"]}
    meta.update({
        "method": "PyMuPDF text spans (font + position); FM-font spans converted with pandukabhaya (fm_abhaya), "
                  "English (Times/Minion) spans kept; lines rebuilt by position (two-column poems, rhyme words); NFC"
                  if b["encoding"] == "fm" else "PyMuPDF text spans (already Unicode); lines rebuilt by position; NFC",
        "quote_fix": "FM glyphs “ and ‘ kept as quote marks before conversion" if b["encoding"] == "fm" else "",
        "pymupdf": md.version("pymupdf"),
        "pandukabhaya": md.version("pandukabhaya"),
        "pages": len(b["pages"]), "sentence_units": len(b["units"]),
        "date": datetime.date.today().isoformat(),
    })
    json.dump(meta, open(f'{b["out"]}/{b["book_id"]}_meta.json', "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(json.dumps(meta, ensure_ascii=False, indent=1))

{
 "book_id": "sinhala_lit_g10_11",
 "pdf": "sinhala lit g-10-11.pdf",
 "title": "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
 "publisher": "Educational Publications Department",
 "grade": "10-11",
 "year": "",
 "url": "",
 "page_offset": 10,
 "encoding": "fm",
 "method": "PyMuPDF text spans (font + position); FM-font spans converted with pandukabhaya (fm_abhaya), English (Times/Minion) spans kept; lines rebuilt by position (two-column poems, rhyme words); NFC",
 "quote_fix": "FM glyphs “ and ‘ kept as quote marks before conversion",
 "pymupdf": "1.28.2",
 "pandukabhaya": "1.0.1",
 "pages": 140,
 "sentence_units": 3190,
 "date": "2026-10-14"
}


9. Sentence extraction with base word

In [ ]:
import openpyxl
import pandas as pd
from collections import defaultdict

WORDS_XLSX = f"{BASE}/sinhala_polysemy_words.xlsx"      # sheet "words": No | Word | forms (rule-based)
REMOVED    = {"වේ"}                                    # words dropped from the study
EXPECTED_N = 67

def load_words(path, sheet="words"):
    """-> [(No, word cell, [spellings])] in sheet order, without REMOVED words."""
    rows = openpyxl.load_workbook(path)[sheet].iter_rows(values_only=True)
    header = [str(h).strip() if h else "" for h in next(rows)]           # 'No ' has a trailing space
    i_no, i_word = header.index("No"), header.index("Word")
    out, seen = [], set()
    for r in rows:
        r = tuple(r) + (None,) * len(header)
        if not r[i_word]:
            continue
        word = str(r[i_word]).strip()
        spellings = [s.strip() for s in re.split(r"[/,]", word) if s.strip()]   # "පස / පස්" -> [පස, පස්]
        if word in REMOVED or any(s in REMOVED for s in spellings):
            continue
        if word in seen:
            raise ValueError(f"Duplicate word in sheet: {word}")
        seen.add(word)
        out.append((int(float(r[i_no])), word, spellings))
    return out

WORDS = load_words(WORDS_XLSX)
assert len(WORDS) == EXPECTED_N, f"Expected {EXPECTED_N} words, got {len(WORDS)}"

JOINERS = re.compile(r"[\u200C\u200D]")
TOKEN   = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")
key     = lambda t: JOINERS.sub("", unicodedata.normalize("NFC", t))   # ZWJ-insensitive comparison

SPELLING2WORD = {}                                     # spelling -> word cell ("පස්" -> "පස / පස්")
for no, word, spellings in WORDS:
    for s in spellings:
        if key(s) in SPELLING2WORD and SPELLING2WORD[key(s)] != word:
            raise ValueError(f"Spelling {s} belongs to two words")
        SPELLING2WORD[key(s)] = word
word_no = {w: no for no, w, _ in WORDS}
print(f"{len(WORDS)} words | {len(SPELLING2WORD)} spellings (two-spelling words counted once each)")

for b in BOOKS:
    if "units" not in b: continue
    rows = []
    for u in b["units"]:
        found = {}                                     # word -> spelling actually seen in this unit
        for tok in TOKEN.findall(u["text"]):
            w = SPELLING2WORD.get(key(tok))
            if w:
                found.setdefault(w, tok)
        for w, tok in found.items():                   # one row per (word, unit), even if it occurs twice
            rows.append({"No": word_no[w], "word": w, "matched": tok, "book_id": b["book_id"],
                         "sid": u["sid"], "pdf_page": u["pdf_page"], "book_page": u["book_page"],
                         "sentence": u["text"]})
    cand = pd.DataFrame(rows, columns=["No", "word", "matched", "book_id", "sid", "pdf_page", "book_page", "sentence"])
    cand = cand.sort_values(["No", "sid"])
    cand.to_csv(f'{b["out"]}/{b["book_id"]}_target_word_sentences.csv', index=False, encoding="utf-8")

    # annotation-tool format (same as extracted_nsina_sentences.txt); ';' inside a sentence -> ','
    with open(f'{b["out"]}/{b["book_id"]}_target_word_sentences.txt', "w", encoding="utf-8") as f:
        f.write("\n\n".join(f"s_word: {w}\nsentences: "
                            + "; ".join(s.replace(";", ",") for s in cand.loc[cand.word == w, "sentence"])
                            for _, w, _ in WORDS))

    summary = pd.DataFrame([{"No": no, "word": w, "spellings": " / ".join(sp),
                             "sentences": int((cand.word == w).sum()),
                             "by_spelling": ", ".join(f"{k} ({v})" for k, v in
                                                      cand.loc[cand.word == w, "matched"].value_counts().items())}
                            for no, w, sp in WORDS])
    summary.to_excel(f'{b["out"]}/{b["book_id"]}_target_word_summary.xlsx', index=False)

    print(f'{b["book_id"]}: {len(cand):,} (word, sentence) rows | {cand.sid.nunique():,} distinct units')
    print("   words with 0 sentences:", list(summary.loc[summary.sentences == 0, "word"]))
    print(summary.sort_values("sentences", ascending=False).head(8)[["No", "word", "sentences", "by_spelling"]].to_string(index=False))

67 words | 72 spellings (two-spelling words counted once each)
sinhala_lit_g10_11: 1,041 (word, sentence) rows | 782 distinct units
   words with 0 sentences: ['වණ', 'රැස', 'කලාපය', 'නිදන', 'දෙස්', 'සමර', 'දල']
 No   word  sentences       by_spelling
 67     වන        150          වන (150)
 55     කර        117          කර (117)
 68     ලද         73           ලද (73)
 59     රජ         65           රජ (65)
  4     ගත         48           ගත (48)
 33    පත්         44          පත් (44)
 69 කල/කල්         40 කල (23), කල් (17)
 30     නද         39           නද (39)


10. Sentence extraction with inflected forms

In [ ]:
import itertools, openpyxl
import pandas as pd
from collections import defaultdict

WORDS_XLSX = f"{BASE}/sinhala_polysemy_words.xlsx"

def load_target_words(path, sheet="words"):
    """-> word list (sheet order), {word: No}, {word: [forms]}. 'පස / පස්' stays ONE word."""
    rows = openpyxl.load_workbook(path)[sheet].iter_rows(values_only=True)
    header = [str(h).strip() if h else "" for h in next(rows)]
    i_no, i_word, i_forms = header.index("No"), header.index("Word"), header.index("Word forms")
    words, numbers, forms = [], {}, {}
    for r in rows:
        r = tuple(r) + (None,) * len(header)
        if not r[i_word]:
            continue
        w = str(r[i_word]).strip()
        if w in numbers:
            raise ValueError(f"Duplicate word in sheet: {w}")
        f = [x.strip() for x in str(r[i_forms] or "").split(",") if x.strip()]
        if not f:
            raise ValueError(f"No forms for {w}")
        words.append(w); numbers[w] = int(float(r[i_no])); forms[w] = list(dict.fromkeys(f))
    return words, numbers, forms

target_words, word_no, word_forms = load_target_words(WORDS_XLSX)
assert len(target_words) == 67, f"Expected 67 words, got {len(target_words)}"

CLITICS = ["ත්", "ුත්", "ද", "ම", "යි", "කි", "වත්", "ය"]      # attach to any form, never change the sense
ENDINGS = [""] + CLITICS + ["".join(p) for p in itertools.product(CLITICS, repeat=2)]
JOINERS = re.compile(r"[\u200C\u200D]")
TOKEN   = re.compile(r"[\u0D80-\u0DFF\u200C\u200D]+")
key     = lambda t: JOINERS.sub("", unicodedata.normalize("NFC", t))   # ZWJ-insensitive comparison

KEY2WORDS, KEY2FORM = defaultdict(set), {}
for w in target_words:
    for f in word_forms[w]:
        for e in ENDINGS:
            k = key(f + e)
            KEY2WORDS[k].add(w)
            KEY2FORM.setdefault((w, k), f)
print(f"{len(target_words)} words | {sum(map(len, word_forms.values())):,} forms | {len(KEY2WORDS):,} token keys")

for b in BOOKS:
    if "units" not in b: continue
    rows, near = [], defaultdict(Counter)
    bases = {w: [key(x.strip()) for x in re.split(r"[/,]", w)] for w in target_words}
    for u in b["units"]:
        hits = {}
        for tok in TOKEN.findall(u["text"]):
            k = key(tok)
            if k in KEY2WORDS:
                for w in KEY2WORDS[k]:
                    hits.setdefault(w, (tok, KEY2FORM[(w, k)]))
            else:   # token that STARTS with a target word but is not one of its forms -> review list
                for w, bs in bases.items():
                    if any(k.startswith(x) and len(k) > len(x) for x in bs):
                        near[w][tok] += 1
        for w, (tok, form) in hits.items():
            rows.append({"No": word_no[w], "word": w, "matched_token": tok, "form": form,
                         "book_id": b["book_id"], "sid": u["sid"], "pdf_page": u["pdf_page"],
                         "book_page": u["book_page"], "sentence": u["text"]})
    cand = pd.DataFrame(rows).sort_values(["No", "sid"])
    cand.to_csv(f'{b["out"]}/{b["book_id"]}_candidates.csv', index=False, encoding="utf-8")


    with open(f'{b["out"]}/{b["book_id"]}_extracted_sentences.txt', "w", encoding="utf-8") as f:
        f.write("\n\n".join(f"s_word: {w}\nsentences: "
                            + "; ".join(s.replace(";", ",") for s in cand.loc[cand.word == w, "sentence"])
                            for w in target_words))

    summary = pd.DataFrame([{
        "No": word_no[w], "word": w,
        "sentences": int((cand.word == w).sum()),
        "forms_found": ", ".join(f"{k} ({v})" for k, v in cand.loc[cand.word == w, "form"].value_counts().items()),
        "not_matched_tokens_starting_with_word (check: missing form or different word?)":
            ", ".join(f"{t} ({c})" for t, c in near[w].most_common(25)),
    } for w in target_words])
    summary.to_excel(f'{b["out"]}/{b["book_id"]}_extraction_summary.xlsx', index=False)

    print(f'{b["book_id"]}: {len(cand):,} (word, sentence) matches | {cand.sid.nunique():,} distinct units')
    print("   words with 0 sentences:", list(summary.loc[summary.sentences == 0, "word"]))
    print(summary[["No", "word", "sentences"]].sort_values("sentences", ascending=False).head(10).to_string(index=False))

67 words | 1,370 forms | 99,724 token keys
sinhala_lit_g10_11: 1,858 (word, sentence) matches | 1,238 distinct units
   words with 0 sentences: ['වණ', 'නිදන', 'සමර', 'දල']
 No   word  sentences
 55     කර        300
 67     වන        159
  4     ගත        120
 69 කල/කල්         92
 68     ලද         78
 59     රජ         73
 22     අත         68
 44    පිය         49
 33    පත්         49
  1    දිය         49
